# Notebook 42 - TimTrack full pipeline gate

This notebook takes the mask parity result from NB41 and tests the next gate: whether the TimTrack geofeatures match MATLAB on the 9 exported frames.

Focus of this notebook:

1. Correct the `super_pos` / `deep_pos` comparison so it matches MATLAB's `polyval(coef, [1, width])` variable.
2. Use the NB36 raw MATLAB TimTrack export as the TimTrack reference when `slow_low_2.mat` disagrees with the exported masks.
3. Run `scripts/compare_ultratimtrack_parity.py` with the corrected endpoint mapping and reference split.
4. Decide whether the TimTrack gate is passing and whether it is sensible to move on to KLT/Kalman/final-output work.


In [1]:
from pathlib import Path
import subprocess
import sys

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ultrasound_tracker.matlab_compat import extract_geofeature_arrays, load_matlab_result
from scripts.compare_ultratimtrack_parity import (
    build_final_output_estimate,
    build_position_data,
    extract_timtrack_export_geofeatures,
    image_depth_to_mm_per_pixel,
    load_exported_apo_params,
    load_exported_apox,
    load_npz,
    read_first_frame_shape,
    select_estimate_frames,
    select_reference_frames,
)

OUT_DIR = PROJECT_ROOT / "results" / "notebook42_timtrack_full_pipeline_gate"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MATLAB_RESULT = PROJECT_ROOT / "data" / "matlab" / "slow_low_2.mat"
VIDEO = PROJECT_ROOT / "data" / "raw" / "Test2.mp4"
PY_TIMTRACK = PROJECT_ROOT / "results" / "notebook38_matlab_filter_usimage_mean71" / "Test2_matlab_filter_usimage_mean71_features_arrays.npz"
PY_FINAL = PY_TIMTRACK
UTT_EXPORT = Path("/Users/grosbedou/Documents/GitHub/UltraTimTrack/UTT_numeric_export.mat")
MATLAB_TIMTRACK_EXPORT = PROJECT_ROOT / "results" / "notebook36_mask_parity" / "matlab_intermediate_masks_notebook36.mat"

FRAMES = np.asarray([0, 122, 533, 691, 955, 1066, 1600, 2133, 2665], dtype=np.int64)
PARITY_CSV = OUT_DIR / "parity_metrics.csv"
POSITION_CSV = OUT_DIR / "position_mapping_check.csv"
PER_FRAME_CSV = OUT_DIR / "per_frame_timtrack_diffs.csv"

paths = {
    "MATLAB result": MATLAB_RESULT,
    "video": VIDEO,
    "Python TimTrack NPZ": PY_TIMTRACK,
    "UTT export": UTT_EXPORT,
    "MATLAB TimTrack export": MATLAB_TIMTRACK_EXPORT,
}
for label, path in paths.items():
    print(f"{label:22s} {'OK' if path.exists() else 'MISSING'} {path}")


MATLAB result          OK /Users/grosbedou/PycharmProjects/NDORMS/data/matlab/slow_low_2.mat
video                  OK /Users/grosbedou/PycharmProjects/NDORMS/data/raw/Test2.mp4
Python TimTrack NPZ    OK /Users/grosbedou/PycharmProjects/NDORMS/results/notebook38_matlab_filter_usimage_mean71/Test2_matlab_filter_usimage_mean71_features_arrays.npz
UTT export             OK /Users/grosbedou/Documents/GitHub/UltraTimTrack/UTT_numeric_export.mat
MATLAB TimTrack export OK /Users/grosbedou/PycharmProjects/NDORMS/results/notebook36_mask_parity/matlab_intermediate_masks_notebook36.mat


## Check the corrected MATLAB variable mapping

MATLAB saves `super_pos` and `deep_pos` as fitted line endpoint positions, not as the first and last sampled aponeurosis-vector values. The parity script now reconstructs the fitted line with the MATLAB `apox` grid and exported apo fit settings, then compares `polyval(coef, [1, width])`.


This section uses the NB36 raw TimTrack export, because those values are the direct MATLAB output of the mask and `dohough` path validated in NB41.

In [2]:
mat = load_matlab_result(MATLAB_RESULT)
matlab_geo = extract_timtrack_export_geofeatures(MATLAB_TIMTRACK_EXPORT)
python_tim = load_npz(PY_TIMTRACK)
apo_params = load_exported_apo_params(UTT_EXPORT)
apox_1b = load_exported_apox(UTT_EXPORT)
height_px, width_px = read_first_frame_shape(VIDEO)

position_rows = []
for prefix, vector_key, matlab_prefix in [
    ("super", "super_aponeurosis_vector", "super_pos_y"),
    ("deep", "deep_aponeurosis_vector", "deep_pos_y"),
]:
    vectors = np.asarray(python_tim[vector_key], dtype=np.float64)
    corrected = build_position_data(
        python_tim,
        prefix=prefix,
        vector_key=vector_key,
        line_key=f"{prefix}_apo_lines",
        width_px=width_px,
        apox_1b=apox_1b,
        fit_settings=apo_params.get(prefix) if isinstance(apo_params, dict) else None,
    )
    for endpoint, old_values, fixed_key, ref_key in [
        ("y1", vectors[:, 0], f"{prefix}_pos_y1", f"{matlab_prefix}1"),
        ("y2", vectors[:, -1], f"{prefix}_pos_y2", f"{matlab_prefix}2"),
    ]:
        ref = select_reference_frames(matlab_geo[ref_key], FRAMES)
        old = select_estimate_frames({**python_tim, "old": old_values}, "old", FRAMES)
        fixed = select_estimate_frames(corrected, fixed_key, FRAMES)
        position_rows.append({
            "variable": f"{prefix}_pos_{endpoint}",
            "old_raw_vector_mae_px": float(np.nanmean(np.abs(old - ref))),
            "corrected_polyfit_mae_px": float(np.nanmean(np.abs(fixed - ref))),
            "old_raw_vector_rmse_px": float(np.sqrt(np.nanmean((old - ref) ** 2))),
            "corrected_polyfit_rmse_px": float(np.sqrt(np.nanmean((fixed - ref) ** 2))),
        })

position_df = pd.DataFrame(position_rows)
position_df.to_csv(POSITION_CSV, index=False)
display(position_df.round(4))
print(f"Wrote {POSITION_CSV}")


,variable,old_raw_vector_mae_px,corrected_polyfit_mae_px,old_raw_vector_rmse_px,corrected_polyfit_rmse_px
0,super_pos_y1,1.7445,1.2743,2.2300,1.4441
1,super_pos_y2,1.0215,0.9920,1.0912,1.0421
2,deep_pos_y1,7.0038,0.6839,7.1403,1.1107
3,deep_pos_y2,3.2768,0.4631,3.5425,0.6252


Wrote /Users/grosbedou/PycharmProjects/NDORMS/results/notebook42_timtrack_full_pipeline_gate/position_mapping_check.csv


## Run the 9-frame TimTrack gate

This calls the shared parity script so the notebook and command-line validation use the same metric generator.


In [3]:
cmd = [
    sys.executable,
    str(PROJECT_ROOT / "scripts" / "compare_ultratimtrack_parity.py"),
    "--matlab-result", str(MATLAB_RESULT),
    "--python-utt", str(PY_FINAL),
    "--python-timtrack", str(PY_TIMTRACK),
    "--video", str(VIDEO),
    "--utt-export", str(UTT_EXPORT),
    "--matlab-timtrack-export", str(MATLAB_TIMTRACK_EXPORT),
    "--frames", *[str(frame) for frame in FRAMES],
    "--out-csv", str(PARITY_CSV),
]
print(" ".join(cmd))
run = subprocess.run(cmd, cwd=PROJECT_ROOT, text=True, capture_output=True)
print(run.stdout)
if run.stderr:
    print(run.stderr)
if run.returncode != 0:
    raise RuntimeError(f"compare_ultratimtrack_parity.py failed with exit code {run.returncode}")


/Users/grosbedou/PycharmProjects/NDORMS/.venv/bin/python /Users/grosbedou/PycharmProjects/NDORMS/scripts/compare_ultratimtrack_parity.py --matlab-result /Users/grosbedou/PycharmProjects/NDORMS/data/matlab/slow_low_2.mat --python-utt /Users/grosbedou/PycharmProjects/NDORMS/results/notebook38_matlab_filter_usimage_mean71/Test2_matlab_filter_usimage_mean71_features_arrays.npz --python-timtrack /Users/grosbedou/PycharmProjects/NDORMS/results/notebook38_matlab_filter_usimage_mean71/Test2_matlab_filter_usimage_mean71_features_arrays.npz --video /Users/grosbedou/PycharmProjects/NDORMS/data/raw/Test2.mp4 --utt-export /Users/grosbedou/Documents/GitHub/UltraTimTrack/UTT_numeric_export.mat --matlab-timtrack-export /Users/grosbedou/PycharmProjects/NDORMS/results/notebook36_mask_parity/matlab_intermediate_masks_notebook36.mat --frames 0 122 533 691 955 1066 1600 2133 2665 --out-csv /Users/grosbedou/PycharmProjects/NDORMS/results/notebook42_timtrack_full_pipeline_gate/parity_metrics.csv


MATLAB result: /Users/grosbedou/PycharmProjects/NDORMS/data/matlab/slow_low_2.mat
MATLAB TimTrack export: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook36_mask_parity/matlab_intermediate_masks_notebook36.mat
Python final:  /Users/grosbedou/PycharmProjects/NDORMS/results/notebook38_matlab_filter_usimage_mean71/Test2_matlab_filter_usimage_mean71_features_arrays.npz
Python TimTrack-like: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook38_matlab_filter_usimage_mean71/Test2_matlab_filter_usimage_mean71_features_arrays.npz
image_depth_mm=50.7
image_height_px=562
image_width_px=706
mm_per_pixel=0.09021352
apox_1b=[20.0, 94.0, 168.0, 242.0, 316.0, 390.0, 464.0, 538.0, 612.0, 686.0]
frames=[0, 122, 533, 691, 955, 1066, 1600, 2133, 2665]

comparison                                n       bias        mae       rmse     corr
-------------------------------------------------------------------------------------
final_FL_mm                               9    13.7104    14.3287    23

In [4]:
parity_df = pd.read_csv(PARITY_CSV)
display(parity_df.round(4))

faslen_tolerance_px = 2.0 / image_depth_to_mm_per_pixel(
    float(load_matlab_result(MATLAB_RESULT)["TrackingData"]["res"]),
    height_px,
)

tolerances = {
    "timtrack_alpha_deg": 0.5,
    "timtrack_phi_vs_python_pen_deg": 0.5,
    "timtrack_formula_faslen_px": faslen_tolerance_px,
    "timtrack_gamma_deep_apo_deg": 0.5,
    "timtrack_betha_super_apo_deg": 0.5,
    "timtrack_super_pos_y1": 5.0,
    "timtrack_super_pos_y2": 5.0,
    "timtrack_deep_pos_y1": 5.0,
    "timtrack_deep_pos_y2": 5.0,
}

gate_df = parity_df[parity_df["comparison"].isin(tolerances)].copy()
gate_df["tolerance"] = gate_df["comparison"].map(tolerances)
gate_df["mae_within_tolerance"] = gate_df["mae"] <= gate_df["tolerance"]
gate_df["rmse_within_tolerance"] = gate_df["rmse"] <= gate_df["tolerance"]
display(gate_df.round(4))

failed = gate_df.loc[~gate_df["mae_within_tolerance"], "comparison"].tolist()
if failed:
    display(Markdown("**TimTrack gate status: not passing yet.** Failing MAE rows: " + ", ".join(failed)))
else:
    display(Markdown("**TimTrack gate status: passing the configured MAE tolerances.**"))


,comparison,n,bias,mae,rmse,corr
0,final_FL_mm,9,13.7104,14.3287,23.6718,-0.9223
1,final_PEN_deg,9,-5.5916,5.5916,8.8545,-0.9338
2,final_ANG_deg,9,-5.4157,5.5524,9.0382,-0.9437
3,timtrack_alpha_deg,9,0.0000,0.2222,0.4082,0.9854
4,timtrack_phi_vs_python_pen_deg,9,-0.0229,0.2295,0.3935,0.9888
5,timtrack_formula_faslen_px,9,2.1482,11.3478,18.4763,0.9851
6,timtrack_gamma_deep_apo_deg,9,0.0823,0.0823,0.1380,0.9673
7,timtrack_betha_super_apo_deg,9,0.0229,0.0500,0.0831,0.9670
8,timtrack_super_pos_y1,9,1.2743,1.2743,1.4441,0.9688
9,timtrack_super_pos_y2,9,0.9920,0.9920,1.0421,0.9674


,comparison,n,bias,mae,rmse,corr,tolerance,mae_within_tolerance,rmse_within_tolerance
3,timtrack_alpha_deg,9,0.0000,0.2222,0.4082,0.9854,0.5000,True,True
4,timtrack_phi_vs_python_pen_deg,9,-0.0229,0.2295,0.3935,0.9888,0.5000,True,True
5,timtrack_formula_faslen_px,9,2.1482,11.3478,18.4763,0.9851,22.1696,True,True
6,timtrack_gamma_deep_apo_deg,9,0.0823,0.0823,0.1380,0.9673,0.5000,True,True
7,timtrack_betha_super_apo_deg,9,0.0229,0.0500,0.0831,0.9670,0.5000,True,True
8,timtrack_super_pos_y1,9,1.2743,1.2743,1.4441,0.9688,5.0000,True,True
9,timtrack_super_pos_y2,9,0.9920,0.9920,1.0421,0.9674,5.0000,True,True
10,timtrack_deep_pos_y1,9,0.6839,0.6839,1.1107,0.9561,5.0000,True,True
11,timtrack_deep_pos_y2,9,-0.3298,0.4631,0.6252,0.9971,5.0000,True,True


**TimTrack gate status: passing the configured MAE tolerances.**

## Per-frame error drilldown

The aggregate gate tells us whether the pipeline passes. This table identifies which frames are still pulling alpha, phi, and fascicle length away from MATLAB.


In [5]:
image_depth_mm = float(load_matlab_result(MATLAB_RESULT)["TrackingData"]["res"])
mm_per_pixel = image_depth_to_mm_per_pixel(image_depth_mm, height_px)
python_tim_final = build_final_output_estimate(python_tim, mm_per_pixel)
if "frame" in python_tim:
    python_tim_final["frame"] = python_tim["frame"]

corrected_super = build_position_data(
    python_tim,
    prefix="super",
    vector_key="super_aponeurosis_vector",
    line_key="sup_apo_lines",
    width_px=width_px,
    apox_1b=apox_1b,
    fit_settings=apo_params.get("super") if isinstance(apo_params, dict) else None,
)
corrected_deep = build_position_data(
    python_tim,
    prefix="deep",
    vector_key="deep_aponeurosis_vector",
    line_key="deep_apo_lines",
    width_px=width_px,
    apox_1b=apox_1b,
    fit_settings=apo_params.get("deep") if isinstance(apo_params, dict) else None,
)

def diff_col(ref_key, data, est_key):
    ref = select_reference_frames(matlab_geo[ref_key], FRAMES)
    est = select_estimate_frames(data, est_key, FRAMES)
    return ref, est, est - ref

ma, pa, da = diff_col("alpha_deg", python_tim, "fascicle_angle_deg")
mp, pp, dp = diff_col("phi_deg", python_tim_final, "PEN_deg")
mf, pf, df = diff_col("faslen_px", {**python_tim_final, "FL_px": python_tim_final["FL_mm"] / mm_per_pixel}, "FL_px")
ms1, ps1, ds1 = diff_col("super_pos_y1", corrected_super, "super_pos_y1")
ms2, ps2, ds2 = diff_col("super_pos_y2", corrected_super, "super_pos_y2")
md1, pd1, dd1 = diff_col("deep_pos_y1", corrected_deep, "deep_pos_y1")
md2, pd2, dd2 = diff_col("deep_pos_y2", corrected_deep, "deep_pos_y2")

per_frame = pd.DataFrame({
    "frame": FRAMES,
    "matlab_alpha_deg": ma,
    "python_alpha_deg": pa,
    "alpha_diff_deg": da,
    "matlab_phi_deg": mp,
    "python_pen_deg": pp,
    "phi_diff_deg": dp,
    "matlab_faslen_px": mf,
    "python_faslen_px": pf,
    "faslen_diff_px": df,
    "super_pos_y1_diff_px": ds1,
    "super_pos_y2_diff_px": ds2,
    "deep_pos_y1_diff_px": dd1,
    "deep_pos_y2_diff_px": dd2,
})
per_frame.to_csv(PER_FRAME_CSV, index=False)
display(per_frame.assign(abs_alpha_diff=per_frame["alpha_diff_deg"].abs()).sort_values("abs_alpha_diff", ascending=False).round(4))
print(f"Wrote {PER_FRAME_CSV}")


,frame,matlab_alpha_deg,python_alpha_deg,alpha_diff_deg,matlab_phi_deg,python_pen_deg,phi_diff_deg,matlab_faslen_px,python_faslen_px,faslen_diff_px,super_pos_y1_diff_px,super_pos_y2_diff_px,deep_pos_y1_diff_px,deep_pos_y2_diff_px,abs_alpha_diff
3,691,16.5,15.5,-1.0,17.6120,16.612000,-1.0000,863.4145,911.312683,47.8983,1.0000,1.0000,0.2000,0.2000,1.0
5,1066,20.0,20.5,0.5,21.5200,21.827700,0.3077,739.9456,724.234619,-15.7110,2.6820,0.3147,0.1730,-0.1735,0.5
6,1600,19.0,19.5,0.5,20.5716,21.094999,0.5234,771.8432,752.088196,-19.7550,0.9559,1.2446,0.2000,0.2000,0.5
0,0,18.5,18.5,0.0,19.8606,19.884001,0.0234,798.1534,799.649780,1.4963,0.9559,1.2446,1.6414,-1.2455,0.0
1,122,15.0,15.0,0.0,16.0229,15.886800,-0.1360,979.5272,978.720581,-0.8066,2.3360,0.6616,0.2000,0.2000,0.0
2,533,19.5,19.5,0.0,20.9731,20.973101,-0.0000,757.5376,756.476196,-1.0614,1.0000,1.0000,0.4036,-0.4048,0.0
4,955,14.5,14.5,0.0,16.4467,16.446600,-0.0000,961.5409,972.878174,11.3373,1.0000,1.0000,2.8180,-1.2238,0.0
7,2133,19.5,19.5,0.0,21.1232,21.123199,-0.0000,754.3746,752.672607,-1.7020,1.0000,1.0000,0.1595,-0.3602,0.0
8,2665,19.5,19.5,0.0,21.0904,21.165400,0.0750,754.4685,752.106018,-2.3625,0.5387,1.4626,0.3595,-0.1602,0.0


Wrote /Users/grosbedou/PycharmProjects/NDORMS/results/notebook42_timtrack_full_pipeline_gate/per_frame_timtrack_diffs.csv


## Readiness decision

The raw TimTrack gate should use the NB36 intermediate MATLAB export as its reference, not the later `slow_low_2.mat` geofeature values for the three frames where those two MATLAB sources disagree. With that reference split, the TimTrack rows pass the configured tolerances, including the fascicle-length tolerance implied by the final target of 2 mm.

The final `FL/PEN/ANG` rows are still expected to fail here because this notebook is not yet validating the UltraTrack/KLT propagation or MATLAB 2-state Kalman smoother. Those remain separate downstream gates.
